In [1]:
# Importações Qiskit 2.3
from qiskit import QuantumCircuit
from qiskit_ibm_runtime import EstimatorV2, SamplerV2
from qiskit.quantum_info import SparsePauliOp
from qiskit_algorithms.optimizers import COBYLA
from qiskit_aer import AerSimulator

from typing import List, Tuple, Dict, Optional
from itertools import permutations

import time
import numpy as np

In [2]:
class QAOATSP:    
    def __init__(self, n_vertices: int, distances: Optional[np.ndarray] = None, seed: int = 42):
        """
        Args:
            num_vertices: Número de cidades
            distances: Matriz de distâncias (nxn). Se None, gera aleatória.
            seed: Seed para reproducibilidade
        """
        self.n_vertices = n_vertices
        self.distances = distances
        self.seed = seed
        np.random.seed(seed)

        # Para TSP: precisamos de uma variável por aresta única (não direcionada)
        # Matriz triangular superior: n(n-1)/2
        self.n_edges = self.n_vertices * (self.n_vertices - 1) // 2
        self.n_qubits = self.n_edges
        
        # Mapear índice de qubit para aresta (i, j)
        self.edge_map = {}
        self.reverse_edge_map = {}
        qubit_idx = 0
        for i in range(self.n_vertices):
            for j in range(i + 1, self.n_vertices):
                self.edge_map[qubit_idx] = (i, j)
                self.reverse_edge_map[(i, j)] = qubit_idx
                self.reverse_edge_map[(j, i)] = qubit_idx
                qubit_idx += 1

        self.simulator = AerSimulator()
        self.estimator = EstimatorV2(mode=self.simulator)
        self.sampler = SamplerV2(mode=self.simulator)

    def _build_cost_hamiltonian(self) -> SparsePauliOp:
        pauli_dict = {}
        
        for qubit, (i, j) in self.edge_map.items():
            weight = self.distances[i, j]
            
            # Termo Z
            pauli_str = ['I'] * self.n_qubits
            pauli_str[qubit] = 'Z'
            pauli_str = ''.join(pauli_str)
            
            if pauli_str in pauli_dict:
                pauli_dict[pauli_str] -= weight / 2
            else:
                pauli_dict[pauli_str] = -weight / 2
        
        return SparsePauliOp.from_list([(k, v) for k, v in pauli_dict.items()]).simplify()
    
    def _build_constraint_hamiltonian(self, penalty_weight: float = None) -> SparsePauliOp:
        if penalty_weight is None:
            avg_edge = float(np.mean(self.distances[self.distances > 0]))
            penalty_weight = avg_edge * self.n_vertices * 1.5  

        pauli_dict = {}
        num_qubits = self.n_qubits

        for v in range(self.n_vertices):
            edges_v = [q for q, (i, j) in self.edge_map.items() if i == v or j == v]
            k = len(edges_v)

            # --- Termos lineares (coeficiente correto com sinal) ---
            linear_coeff = penalty_weight * (2.0 - k / 2.0)  

            for q in edges_v:
                pauli_str = ['I'] * num_qubits
                pauli_str[q] = 'Z'
                p_str = "".join(pauli_str[::-1])  # convenção Qiskit
                pauli_dict[p_str] = pauli_dict.get(p_str, 0.0) + linear_coeff

            # --- Termos quadráticos (interação par a par) ---
            for idx, q1 in enumerate(edges_v):
                for q2 in edges_v[idx + 1:]:
                    pauli_str = ['I'] * num_qubits
                    pauli_str[q1] = 'Z'
                    pauli_str[q2] = 'Z'
                    p_str = "".join(pauli_str[::-1])
                    pauli_dict[p_str] = pauli_dict.get(p_str, 0.0) + penalty_weight / 2.0

        return SparsePauliOp.from_list(
            [(k, v) for k, v in pauli_dict.items() if abs(v) > 1e-12]
        ).simplify()
    
    def _build_xy_mixer_hamiltonian(self) -> SparsePauliOp:
        pauli_list = []
        for q1 in range(self.n_qubits):
            for q2 in range(q1 + 1, self.n_qubits):
                edge1 = self.edge_map[q1]
                edge2 = self.edge_map[q2]
                
                if any(v in edge2 for v in edge1):
                    x_str = list('I' * self.n_qubits)
                    x_str[q1] = 'X'
                    x_str[q2] = 'X'
                    pauli_list.append(("".join(x_str), 0.5))
                    
                    y_str = list('I' * self.n_qubits)
                    y_str[q1] = 'Y'
                    y_str[q2] = 'Y'
                    pauli_list.append(("".join(y_str), 0.5))

        return SparsePauliOp.from_list(pauli_list).simplify()  
      
    def _get_total_hamiltonian(self) -> SparsePauliOp:
        H_cost = self._build_cost_hamiltonian()
        H_res = self._build_constraint_hamiltonian()
        return (H_cost + H_res).simplify()
      
    def _apply_warm_start(self, qc: QuantumCircuit):
        # Encontra um tour simples (ex: vizinho mais próximo)
        curr = 0
        visited = {0}
        tour_edges = []
        
        for _ in range(self.n_vertices - 1):
            # Encontra o vizinho mais próximo não visitado
            next_v = min([v for v in range(self.n_vertices) if v not in visited],
                        key=lambda v: self.distances[curr, v])
            tour_edges.append(tuple(sorted((curr, next_v))))
            visited.add(next_v)
            curr = next_v
        
        # Fecha o ciclo
        tour_edges.append(tuple(sorted((curr, 0))))
        
        # Aplica no circuito
        for edge in tour_edges:
            qubit = self.reverse_edge_map[edge]
            qc.x(qubit)    

    def _create_qaoa_circuit(self, params: np.ndarray) -> QuantumCircuit:
        p = len(params) // 2
        beta_params = params[:p]
        gamma_params = params[p:]
        
        qc = QuantumCircuit(self.n_qubits)

		# Warm-Start
        self._apply_warm_start(qc)

        # Aplicar p camadas de QAOA
        for layer in range(p):
            # Camada de custo
            #cost_h = self._build_cost_hamiltonian()
            cost_h = self._get_total_hamiltonian()
            self._apply_hamiltonian(qc, cost_h, gamma_params[layer])
            
            # Camada de mixer
            #mixer_h = self._build_mixer_hamiltonian()
            mixer_h = self._build_xy_mixer_hamiltonian()
            self._apply_hamiltonian(qc, mixer_h, beta_params[layer])

        return qc
    
    def _apply_hamiltonian(self, qc: QuantumCircuit, 
                          hamiltonian: SparsePauliOp, 
                          time: float) -> None:
        for pauli_str, coeff in hamiltonian.to_list():
            coeff_real = float(np.real(coeff))
            if abs(coeff_real) < 1e-10:
                continue

            # pauli_str em ordem Qiskit: índice 0 = qubit mais à direita
            pauli_rev = pauli_str[::-1]  # pauli_rev[i] = operador no qubit i
            active = [(i, p) for i, p in enumerate(pauli_rev) if p != 'I']

            if not active:
                continue  # fase global — ignorar

            indices = [i for i, _ in active]
            ops     = {i: p for i, p in active}

            # 1. Rotação de base: X→Z (via H), Y→Z (via S†H)
            for i, p in active:
                if p == 'X':
                    qc.h(i)
                elif p == 'Y':
                    qc.sdg(i)
                    qc.h(i)
                # Z: não precisa de rotação

            # 2. Cadeia de CNOTs para computar a paridade
            for k in range(len(indices) - 1):
                qc.cx(indices[k], indices[k + 1])

            # 3. Rotação RZ no último qubit (único ponto com ângulo)
            qc.rz(2.0 * coeff_real * time, indices[-1])

            # 4. Desfaz a cadeia de CNOTs (ordem inversa)
            for k in range(len(indices) - 2, -1, -1):
                qc.cx(indices[k], indices[k + 1])

            # 5. Desfaz rotação de base
            for i, p in active:
                if p == 'X':
                    qc.h(i)
                elif p == 'Y':
                    qc.h(i)
                    qc.s(i)

    def _interp_params(self, p_prev: int, params_prev: np.ndarray, p_new: int) -> np.ndarray:
        beta_prev  = params_prev[:p_prev]
        gamma_prev = params_prev[p_prev:]

        def interp(arr, new_len):
            x_old = np.linspace(0, 1, len(arr))
            x_new = np.linspace(0, 1, new_len)
            return np.interp(x_new, x_old, arr)

        return np.concatenate([interp(beta_prev, p_new), interp(gamma_prev, p_new)])

    def optimize(self, p: int = 1, max_iter: int = 100, num_restarts: int = 5,
                use_interp: bool = True, debug: bool = False) -> Dict:
        best_result = None
        best_energy = float('inf')

        for restart in range(num_restarts):
            if debug:
            	print(f"\n[Restart {restart+1}/{num_restarts}]")

            # Estratégia INTERP: escala gradual de p=1 até p alvo
            if use_interp and p > 1:
                # Começa com p=1 para obter bom ponto inicial
                gamma_init = np.random.uniform(0, np.pi / 4, 1)
                beta_init  = np.random.uniform(0, np.pi / 2, 1)
                current_params = np.concatenate([beta_init, gamma_init])
                current_p = 1

                while current_p < p:
                    def _obj(params, _p=current_p):
                        try:
                            qc = self._create_qaoa_circuit_p(params, _p)
                            job = self.estimator.run([(qc, self._get_total_hamiltonian())])
                            return float(job.result()[0].data.evs)
                        except:
                            return 1e10

                    opt = COBYLA(maxiter=max_iter // 2, tol=1e-3)
                    res = opt.minimize(_obj, current_params)
                    current_p += 1
                    current_params = self._interp_params(current_p - 1, res.x, current_p)

                initial_params = current_params
            else:
                gamma_init = np.random.uniform(0, np.pi / 2, p)
                beta_init  = np.random.uniform(0, np.pi,     p)
                initial_params = np.concatenate([beta_init, gamma_init])

            iteration = [0]
            energy_history = []

            def objective_function(params):
                iteration[0] += 1
                try:
                    qc = self._create_qaoa_circuit(params)
                    job = self.estimator.run([(qc, self._get_total_hamiltonian())])
                    energy = float(job.result()[0].data.evs)
                    energy_history.append(energy)
                    if iteration[0] % 20 == 0 and debug:
                        print(f"  Iter {iteration[0]}: E = {energy:.4f}")
                    return energy
                except:
                    return 1e10

            optimizer = COBYLA(maxiter=max_iter, tol=1e-4)
            result = optimizer.minimize(objective_function, initial_params)

            if result.fun < best_energy:
                best_energy = result.fun
                best_result = {
                    'optimal_params': result.x,
                    'optimal_energy': result.fun,
                    'iterations': iteration[0],
                    'energy_history': energy_history,
                    'num_params': 2 * p,
                }
                if debug:
                	print(f"Novo melhor: {best_energy:.4f}")

        return best_result
    
    def _two_opt(self, tour: List[int]) -> List[int]:
        best = tour[:]
        n = len(best)
        improved = True
        while improved:
            improved = False
            for i in range(1, n - 1):
                for j in range(i + 1, n):
                    new_tour = best[:i] + best[i:j+1][::-1] + best[j+1:]
                    if self._calculate_tour_distance(new_tour) < self._calculate_tour_distance(best):
                        best = new_tour
                        improved = True
        return best
    
    def get_solution(self, params: np.ndarray, num_shots: int = 1024) -> Tuple[List, float, Dict]:
        qc = self._create_qaoa_circuit(params)
        qc.measure_all()

        result = self.sampler.run([qc], shots=num_shots).result()
        counts = result[0].data.meas.get_counts()
        
        unique_tours = {}
        for bitstring, count in counts.items():
            tour = self._bitstring_to_tour(bitstring)
            tour = self._two_opt(tour)   
            norm_tour = self._normalize_tour(tour)
            tour_key = tuple(norm_tour)
            
            if tour_key not in unique_tours:
                unique_tours[tour_key] = {
                    'count': count, 
                    'distance': self._calculate_tour_distance(norm_tour)
                }
            else:
                unique_tours[tour_key]['count'] += count
                
        sorted_by_distance = sorted(unique_tours.items(), key=lambda x: x[1]['distance'])
        best_tour, best_data = sorted_by_distance[0]
        
        stats = {
            'probability': best_data['count'] / num_shots,
            'top_by_distance': sorted_by_distance[:10] # Top rotas mais curtas encontradas
        }
        return list(best_tour), best_data['distance'], stats

    def _bitstring_to_tour(self, bitstring: str) -> List[int]:
        # Bitstring invertido para correspondência com ordem de qubits
        bitstring = bitstring[::-1]
        
        # Iniciar do vértice 0
        tour = [0]
        visited = {0}
        current = 0
        
        # Construir tour guloso baseado em bits ativados
        edges_set = set()
        for qubit, bit in enumerate(bitstring):
            if bit == '1':
                i, j = self.edge_map[qubit]
                edges_set.add((min(i, j), max(i, j)))
        
        # Construir tour a partir das arestas
        while len(visited) < self.n_vertices:
            found_next = False
            for i, j in edges_set:
                if i == current and j not in visited:
                    tour.append(j)
                    visited.add(j)
                    current = j
                    found_next = True
                    break
                elif j == current and i not in visited:
                    tour.append(i)
                    visited.add(i)
                    current = i
                    found_next = True
                    break
            
            if not found_next:
                # Adicionar vértice não visitado mais próximo
                remaining = [v for v in range(self.n_vertices) if v not in visited]
                if remaining:
                    nearest = min(remaining, key=lambda v: self.distances[current, v])
                    tour.append(nearest)
                    visited.add(nearest)
                    current = nearest
        
        return tour
    
    def _normalize_tour(self, tour: List[int]) -> List[int]:
        """Normaliza o tour para que sempre comece pelo menor índice e tenha direção única."""
        if not tour: return []
        idx = tour.index(min(tour))
        shifted = tour[idx:] + tour[:idx]
        if len(shifted) > 2 and shifted[1] > shifted[-1]:
            shifted = [shifted[0]] + shifted[:0:-1]
        return shifted

    def _bitstring_to_tour(self, bitstring: str) -> List[int]:
        bitstring = bitstring[::-1]
        edges = [self.edge_map[i] for i, bit in enumerate(bitstring) if bit == '1']
        
        adj = {i: [] for i in range(self.n_vertices)}
        for u, v in edges:
            adj[u].append(v)
            adj[v].append(u)
            
        tour = [0]
        visited = {0}
        curr = 0
        while len(visited) < self.n_vertices:
            options = [v for v in adj[curr] if v not in visited]
            if not options: # Fallback guloso se o grafo for desconexo
                remaining = [v for v in range(self.n_vertices) if v not in visited]
                next_v = min(remaining, key=lambda x: self.distances[curr, x])
            else:
                next_v = options[0]
            tour.append(next_v)
            visited.add(next_v)
            curr = next_v
        return tour

    def _calculate_tour_distance(self, tour):
        return sum(self.distances[tour[i], tour[(i+1)%len(tour)]] for i in range(len(tour)))
    
    def solve_classical(self) -> Tuple[List, float]:
        """Solução clássica por força bruta (para comparação)"""
        start = time.perf_counter_ns()
        best_tour = None
        best_distance = float('inf')
        
        for perm in permutations(range(1, self.n_vertices)):
            tour = [0] + list(perm)
            distance = self._calculate_tour_distance(tour)
            if distance < best_distance:
                best_distance = distance
                best_tour = tour

        return best_tour, best_distance, time.perf_counter_ns() - start 

In [ ]:
# Formatação tempo
def format(ns):
  horas = ns // (3600 * 10**9)
  minutos = (ns // (60 * 10**9)) % 60
  segundos = (ns // 10**9) % 60
  milissegundos = (ns // 10**6) % 1000
  nanosegundos = ns % 10**6

  return f"{horas}h {minutos}m {segundos}s {milissegundos}ms {nanosegundos}ns"

# Geração de matrizes aleatórias
def generate_matrix():
	import numpy as np

	num_vertices = 7
	distances = np.random.randint(1, 20, (num_vertices, num_vertices))
	distances = (distances + distances.T) / 2  
	np.fill_diagonal(distances, 0)
	return distances

In [ ]:
results_summary = {}

experiments = {
	3: { 'p_layer':  3, 'distance': np.array([[ 0.0, 6.5, 11.0 ], 
											   [ 6.5, 0.0, 11.5 ], 
											   [ 11.0, 11.5, 0.0 ] ]) },
  	4: { 'p_layer':  4, 'distance': np.array([[ 0.0,  14.5, 10.0,  9.5 ], 
											   [ 14.5,  0.0, 15.5,  9.5 ], 
											   [ 10.0, 15.5,  0.0,  2.5 ], 
											   [  9.5,  9.5,  2.5,  0.0 ]]) },
	5: { 'p_layer':  6, 'distance': np.array([[  0.0,  11.0,  10.0,  3.0,  10.5 ], 
 											   [ 11.0,   0.0,  10.0,   4.0,  13.0 ],
 											   [ 10.0,  10.0,   0.0,  13.0,   7.5 ],
 											   [  3.0,   4.0,  13.0,   0.0,  16.0 ],
 											   [ 10.5,  13.0,   7.5,  16.0,   0.0 ]]) },
	6: { 'p_layer':  6, 'distance': np.array([[ 0.0,  14.0,  13.5,  4.0,  15.5, 11.5 ],
 											   [ 14.0,  0.0,  14.5,  1.5,  8.5,  8.0 ],
 											   [ 13.5, 14.5,  0.0,  10.5, 12.0,  5.5 ],
											   [ 4.0,   1.5, 10.5,  0.0,  12.0, 7.5],
											   [ 15.5,  8.5, 12.0,  12.0,   0.0,  16.0 ],
											   [ 11.5,  8.0,  5.5, 7.5, 16.0, 0.0 ]]) },
	7: { 'p_layer':  7, 'distance': np.array([[ 0.0,   9.0, 11.0,  10.0,  10.0,  10.0, 5.5],
											   [ 9.0,   0.0, 12.0,  11.0,   6.0,   4.5, 10.5],
											   [11.0,  12.0,  0.0,  10.0,  10.5,  10.0, 10.5],
											   [10.0,  11.0, 10.0,   0.0,  10.0,  10.0, 11.5],
											   [10.0,   6.0, 10.5,  10.0,   0.0,  14.0,  7.5],
											   [10.0,   4.5, 10.0,  10.0,  14.0,   0.0,  4.5],
											   [ 5.5,  10.5, 10.5,  11.5,   7.5,   4.5,  0.0 ]]) }
}

for n_vertices, data in experiments.items():
	p_layer = data['p_layer']
	distance = data['distance']

	start = time.perf_counter_ns()

	opt = QAOATSP(n_vertices=n_vertices, distances=distance)
	opt_result = opt.optimize(p=p_layer, max_iter=400)
	tour, dist, stats = opt.get_solution(opt_result['optimal_params'], num_shots=1024)

	# Obter solução
	tour, distance, stats = opt.get_solution(
		opt_result['optimal_params'], 
		num_shots=1024
	)

	finished = time.perf_counter_ns() - start 

	print(f"SIMULAÇÃO COM {n_vertices} VÉRTICES")
	print("-" * 50)
	print('*** QAOA ***')
	print(f"Qubits necessários: {opt.n_qubits}")
	print(f"Camadas: {p_layer} iterações realizadas: {opt_result['iterations']}")
	print(f"Número de parâmetros otimizados: {opt_result['num_params']}")
	print(f"Rota: {tour} menor distância: {dist:.2f}")
	print(f"Tempo processamento: {format(finished)} ns: {finished}")

	# Solução clássica
	classical_tour, classical_distance, finished = opt.solve_classical()
	print('*** CLÁSSICO ***')
	print(f"Rota: {classical_tour} menor distância: {classical_distance:.2f}")
	print(f"Tempo processamento: {format(finished)} ns: {finished}")

	approximation_ratio = distance / classical_distance
	print(f"\nTaxa de aproximação (QAOA/Clássico): {approximation_ratio:.4f}")
	print(f"Probabilidade da melhor solução: {stats['probability']:.4f}")

	print("\nTop 10 rotas únicas:")
	for i, (t, data) in enumerate(stats['top_by_distance'], 1):
		print(f"{i}. {list(t)} - Dist: {data['distance']:.2f} ({data['count']} counts)")        
		# Armazenar resultados
		results_summary[n_vertices] = {
			'num_qubits': opt.n_qubits,
			'qaoa_distance': distance,
			'classical_distance': classical_distance,
			'approximation_ratio': approximation_ratio,
			'probability': stats['probability'],
			'p_layers': p_layer,
			'energy_history': opt_result['energy_history']
		}
	print("\n")


SIMULAÇÃO COM 3 VÉRTICES
--------------------------------------------------
*** QAOA ***
Qubits necessários: 3
Camadas: 3 iterações realizadas: 37
Número de parâmetros otimizados: 6
Rota: [0, 1, 2] menor distância: 29.00
Tempo processamento: 0h 0m 10s 735ms 864900ns ns: 10735864900
*** CLÁSSICO ***
Rota: [0, 1, 2] menor distância: 29.00
Tempo processamento: 0h 0m 0s 0ms 24600ns ns: 24600

Taxa de aproximação (QAOA/Clássico): 1.0000
Probabilidade da melhor solução: 1.0000

Top 10 rotas únicas:
1. [0, 1, 2] - Dist: 29.00 (1024 counts)


SIMULAÇÃO COM 4 VÉRTICES
--------------------------------------------------
*** QAOA ***
Qubits necessários: 6
Camadas: 4 iterações realizadas: 92
Número de parâmetros otimizados: 8
Rota: [0, 1, 3, 2] menor distância: 36.50
Tempo processamento: 0h 0m 50s 504ms 127700ns ns: 50504127700
*** CLÁSSICO ***
Rota: [0, 1, 3, 2] menor distância: 36.50
Tempo processamento: 0h 0m 0s 0ms 39900ns ns: 39900

Taxa de aproximação (QAOA/Clássico): 1.0000
Probabilidade da 